# 5. evaluation and analyze

Notebook evaluationfine-tuning after OpenVLA-UAV in test above. 

**target: **
- in many above evaluation
- calculate: L1 error, actionaccuracy
- vs true value
- analyze not instructiontype can 
- toward 

## 5.1 environment

In [ ]:
import sys
import os
import json
import time
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from collections import defaultdict
from tqdm.notebook import tqdm

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

PROJECT_ROOT = "/root/autodl-tmp/claude/UAV-Flow/OpenVLA-UAV"
sys.path.insert(0, PROJECT_ROOT)

DATA_DIR = Path("/root/autodl-tmp/claude/data/uav_flow_subset")
CHECKPOINT_DIR = Path("/root/autodl-tmp/claude/runs")

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)

## 5.2 load

In [ ]:
from prismatic.extern.hf.configuration_prismatic import OpenVLAConfig
from prismatic.extern.hf.modeling_prismatic import OpenVLAForActionPrediction
from prismatic.extern.hf.processing_prismatic import PrismaticProcessor, PrismaticImageProcessor
from prismatic.vla.action_tokenizer import ActionTokenizer
from transformers import AutoConfig, AutoImageProcessor, AutoProcessor, AutoModelForVision2Seq

AutoConfig.register("openvla", OpenVLAConfig)
AutoImageProcessor.register(OpenVLAConfig, PrismaticImageProcessor)
AutoProcessor.register(OpenVLAConfig, PrismaticProcessor)
AutoModelForVision2Seq.register(OpenVLAConfig, OpenVLAForActionPrediction)

# checkpoint
checkpoints = [d for d in CHECKPOINT_DIR.iterdir() if d.is_dir() and (d / 'config.json').exists()]
model_path = str(max(checkpoints, key=lambda d: d.stat().st_mtime)) if checkpoints else None
assert model_path, "fine-tuning checkpoint, run Notebook 03 or run_finetune.sh"
print(f" use checkpoint: {model_path}")

processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForVision2Seq.from_pretrained(
model_path, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True, trust_remote_code=True
).to("cuda:0")
model.eval()

stats_path = Path(model_path) / 'dataset_statistics.json'
if stats_path.exists():
with open(stats_path) as f:
model.norm_stats = json.load(f)

action_tokenizer = ActionTokenizer(processor.tokenizer)
print(f"loadcomplete.: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
def predict(image, instruction):
"""inferencefunction"""
prompt = f"In: What action should the robot take to {instruction.lower()}?\nOut:"
inputs = processor(prompt, image).to("cuda:0", dtype=torch.bfloat16)
with torch.no_grad():
gen = model.generate(**inputs, max_new_tokens=4, do_sample=False)
token_ids = gen[0, -4:].cpu().numpy()
norm_action = action_tokenizer.decode_token_ids_to_actions(token_ids)
return norm_action # [-1, 1]

## 5.3 evaluation

from dataset in 20, for each has frameinference, calculate each. 

In [ ]:
# test
np.random.seed(42)
all_trajs = sorted(list(DATA_DIR.iterdir()))
N_EVAL = 20 # evaluation ( few)
eval_trajs = np.random.choice(all_trajs, size=min(N_EVAL, len(all_trajs)), replace=False)

print(f"evaluation: {len(eval_trajs)}")

# actionnormalizationparameter ( and training)
# new calculate or from datasetload
from prismatic.vla.datasets.uav_dataset import SimpleVLADataset
dataset = SimpleVLADataset(
data_path=DATA_DIR,
image_transform=processor.image_processor,
tokenizer=processor.tokenizer,
action_tokenizer=action_tokenizer,
prompt_builder_fn=__import__('prismatic.models.backbones.llm.prompting', fromlist=['PurePromptBuilder']).PurePromptBuilder,
)
action_min = dataset.action_min
action_max = dataset.action_max
print(f"Action min (q01): {action_min}")
print(f"Action max (q99): {action_max}")

In [ ]:
# inference
results = []

for traj_dir in tqdm(eval_trajs, desc="evaluation"):
with open(traj_dir / 'log.json') as f:
log = json.load(f)

instruction = log['instruction']
raw_logs = np.array(log['raw_logs'])
images = sorted(list(traj_dir.glob('*.jpg')))

# calculate true valueaction ( and SimpleVLADataset process)
traj_raw = raw_logs[:, [0,1,2,4]] # x,y,z,pitch -> and dataset 
traj_raw[:, 3] = np.deg2rad(traj_raw[:, 3])

traj_pred_norm = []
traj_gt_norm = []

# for each frameinference ( each 2 framesampling with)
for i in range(0, len(images), 2):
image = Image.open(images[i]).convert("RGB")
pred_norm = predict(image, instruction)
traj_pred_norm.append(pred_norm)

# true value (normalization [-1, 1])
preprocessed = np.array(log['preprocessed_logs'])
gt_4d = preprocessed[i, [0,1,2,5]] # x,y,z,yaw
gt_norm = 2 * (gt_4d - action_min) / (action_max - action_min) - 1
gt_norm = np.clip(gt_norm, -1, 1)
traj_gt_norm.append(gt_norm)

results.append({
'id': log['id'],
'instruction': instruction,
'pred_norm': np.array(traj_pred_norm),
'gt_norm': np.array(traj_gt_norm),
'n_frames': len(images),
})

print(f"\nevaluationcomplete! {len(results)} ")

## 5.4 

In [ ]:
# calculate each 
dim_names = ['X', 'Y', 'Z', 'Yaw']

all_l1 = []
all_mse = []
per_dim_l1 = defaultdict(list)

for r in results:
pred = r['pred_norm']
gt = r['gt_norm']

l1 = np.mean(np.abs(pred - gt))
mse = np.mean((pred - gt) ** 2)
all_l1.append(l1)
all_mse.append(mse)

for d in range(4):
per_dim_l1[dim_names[d]].append(np.mean(np.abs(pred[:, d] - gt[:, d])))

print("=" * 50)
print("evaluationresult (normalization between [-1, 1])")
print("=" * 50)
print(f": {len(results)}")
print(f"\n:")
print(f" Mean L1 Error: {np.mean(all_l1):.4f} (±{np.std(all_l1):.4f})")
print(f" Mean MSE: {np.mean(all_mse):.4f} (±{np.std(all_mse):.4f})")
print(f"\n each L1 Error:")
for d in dim_names:
vals = per_dim_l1[d]
print(f" {d:>4s}: {np.mean(vals):.4f} (±{np.std(vals):.4f})")

In [ ]:
# 
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(all_l1, bins=15, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(np.mean(all_l1), color='red', linestyle='--', label=f'Mean={np.mean(all_l1):.4f}')
axes[0].set_xlabel('L1 Error')
axes[0].set_ylabel('Count')
axes[0].set_title('Per-trajectory L1 Error Distribution')
axes[0].legend()

dim_means = [np.mean(per_dim_l1[d]) for d in dim_names]
dim_stds = [np.std(per_dim_l1[d]) for d in dim_names]
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
axes[1].bar(dim_names, dim_means, yerr=dim_stds, color=colors, edgecolor='white', capsize=5)
axes[1].set_ylabel('Mean L1 Error')
axes[1].set_title('Per-dimension L1 Error')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 5.5: vs true value

In [ ]:
# 4 for 
fig, axes = plt.subplots(4, 4, figsize=(16, 12))

for row, r in enumerate(results[:4]):
for col, dim in enumerate(dim_names):
ax = axes[row, col]
t = range(len(r['gt_norm']))
ax.plot(t, r['gt_norm'][:, col], 'b-', linewidth=1.5, label='GT' if row == 0 else '')
ax.plot(t, r['pred_norm'][:, col], 'r--', linewidth=1.5, label='Pred' if row == 0 else '')
ax.grid(True, alpha=0.2)

if row == 0:
ax.set_title(dim, fontsize=12)
ax.legend(fontsize=8)
if col == 0:
ax.set_ylabel(r['instruction'][:25] + '...', fontsize=8)
if row == 3:
ax.set_xlabel('Frame')

plt.suptitle('Trajectory Comparison: Ground Truth (blue) vs Predicted (red)', fontsize=14)
plt.tight_layout()
plt.show()

## 5.6 instructiontypeanalyze

In [ ]:
# by instruction in key
categories = {
'left': [],
'right': [],
'front': [],
'toward': [],
}

for i, r in enumerate(results):
inst = r['instruction'].lower()
for key in categories:
if key in inst:
categories[key].append(all_l1[i])

print(" by instructionkey L1 Error:")
print("-" * 40)
cat_names = []
cat_means = []
cat_stds = []
for key, vals in categories.items():
if vals:
cat_names.append(key)
cat_means.append(np.mean(vals))
cat_stds.append(np.std(vals))
print(f" '{key}': n={len(vals):>3d}, L1={np.mean(vals):.4f} (±{np.std(vals):.4f})")

if cat_names:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(cat_names, cat_means, yerr=cat_stds, color=['#4C72B0', '#55A868', '#C44E52', '#8172B2'],
edgecolor='white', capsize=5)
ax.set_ylabel('Mean L1 Error')
ax.set_title('Performance by Instruction Type')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 5.7 errorgraph: frame? 

In [ ]:
# analyzeerrorframeposition (normalization [0, 1] for position)
position_errors = defaultdict(list)

for r in results:
n = len(r['gt_norm'])
for i in range(n):
rel_pos = i / max(n - 1, 1) # 0=, 1=end
bucket = int(rel_pos * 10) / 10 # 10 bucket
err = np.mean(np.abs(r['pred_norm'][i] - r['gt_norm'][i]))
position_errors[bucket].append(err)

buckets = sorted(position_errors.keys())
means = [np.mean(position_errors[b]) for b in buckets]
stds = [np.std(position_errors[b]) for b in buckets]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar([f"{b:.0%}" for b in buckets], means, yerr=stds, color='steelblue', 
edgecolor='white', capsize=3, alpha=0.8)
ax.set_xlabel('Relative Position in Trajectory')
ax.set_ylabel('Mean L1 Error')
ax.set_title('Prediction Error vs Trajectory Position\n(Start=0%, End=100%)')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(": /endframeerror small (action 0), in between frameaction large,. ")

## 5.8 and toward 

### before features
- use 500 LoRA fine-tuning, in 500 above training
- 4 action (x, y, z, yaw)

### can toward 

| toward | method | |
|------|------|----------|
| **data** | use all 30K | generalization can |
| **training** | 2000-5000 | loss convergence |
| **History frames** | input many frameimage | |
| **Action chunking** | times many action | few error |
| ** large LoRA rank** | r=64 or r=128 | can |
| **data** | image, | robustness |
| **Curriculum learning** | after | convergence |

In [ ]:
# 
del model, dataset
torch.cuda.empty_cache()
print("evaluationcomplete,.")

## small 

Notebook complete UAV-Flow VLN process:

1. **data exploration** - dataset, and action space
2. **model architecture** - encoder + LLM + Action Tokenizer
3. **LoRA fine-tuning** - training 1% parameter i.e.dronetask
4. **inference** - image + instruction → action
5. **evaluationanalyze** - + + erroranalyze

this large pre-training-language model controltask above. 